# Parte A — Fine-tuning dei checkpoint pretrainati su ESC-50
### Verifica trend: acc_75 > acc_50 > acc_25

**Prerequisito**: aver eseguito `PartA_pretrain.ipynb` e avere su Drive:
- `mae_ast_pretrained_25pct_final.pt`
- `mae_ast_pretrained_50pct_final.pt`
- `mae_ast_pretrained_75pct_final.pt`

**Pipeline per ogni checkpoint**:
1. Converti fairseq → timm con `scripts/convert_checkpoint.py`
2. Fine-tuning 5-fold su ESC-50 con `src/train.py`
3. Confronta accuracy tra i tre masking ratio

In [ ]:
# CELLA 0 — Setup
import torch
import os, sys, json

if not torch.cuda.is_available():
    raise SystemExit('GPU non trovata')
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install timm==0.9.16 librosa==0.10.1 --quiet

REPO_URL = 'https://github.com/alessiopgg/deep_learning_project.git'
REPO_DIR = '/content/mae-ast-poggesi'
if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull --quiet')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR} --quiet')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/mae_ast_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup completo')

In [ ]:
# CELLA 1 — Verifica checkpoint pretrainati e converti
# I checkpoint del pretraining usano il formato del nostro SimpleMAEAST,
# che va convertito in formato timm per il fine-tuning con src/model.py.

MASK_RATIOS = [25, 50, 75]
checkpoints_pretrained = {}
checkpoints_converted = {}

for pct in MASK_RATIOS:
    # Cerca checkpoint pretrainato
    for suffix in ['_final.pt', '.pt']:
        ck_path = os.path.join(RESULTS_DIR, f'mae_ast_pretrained_{pct}pct{suffix}')
        if os.path.exists(ck_path):
            checkpoints_pretrained[pct] = ck_path
            break

    if pct not in checkpoints_pretrained:
        print(f'MANCANTE: checkpoint {pct}% non trovato in {RESULTS_DIR}')
        continue

    mb = os.path.getsize(checkpoints_pretrained[pct]) / 1e6
    print(f'OK {pct}%: {checkpoints_pretrained[pct]}  ({mb:.0f} MB)')

print(f'\nCheckpoint trovati: {len(checkpoints_pretrained)}/{len(MASK_RATIOS)}')

In [ ]:
# CELLA 2 — Conversione checkpoint: SimpleMAEAST -> timm ViT
#
# Il modello SimpleMAEAST del pretraining usa nn.TransformerEncoder,
# mentre il fine-tuning usa timm ViT. Convertiamo le chiavi.

def convert_simple_to_timm(input_path, output_path):
    """Converte checkpoint SimpleMAEAST -> formato timm ViT-base."""
    ck = torch.load(input_path, map_location='cpu')
    state_in = ck['model']

    state_out = {}

    # post_extract_proj -> patch_embed.proj
    if 'post_extract_proj.weight' in state_in:
        w = state_in['post_extract_proj.weight']  # [768, 256]
        state_out['patch_embed.proj.weight'] = w.reshape(768, 1, 16, 16)
    if 'post_extract_proj.bias' in state_in:
        state_out['patch_embed.proj.bias'] = state_in['post_extract_proj.bias']

    # Encoder layers: nn.TransformerEncoder -> timm blocks
    # nn.TransformerEncoder: encoder.layers.N.{self_attn, linear1, linear2, norm1, norm2}
    # timm: blocks.N.{attn.qkv, attn.proj, mlp.fc1, mlp.fc2, norm1, norm2}
    for n in range(12):
        p_in = f'encoder.layers.{n}.'
        p_out = f'blocks.{n}.'

        # Check if layer exists
        if f'{p_in}self_attn.in_proj_weight' not in state_in:
            continue

        # self_attn.in_proj_{weight,bias} -> attn.qkv.{weight,bias}
        if f'{p_in}self_attn.in_proj_weight' in state_in:
            state_out[f'{p_out}attn.qkv.weight'] = state_in[f'{p_in}self_attn.in_proj_weight']
        if f'{p_in}self_attn.in_proj_bias' in state_in:
            state_out[f'{p_out}attn.qkv.bias'] = state_in[f'{p_in}self_attn.in_proj_bias']

        # self_attn.out_proj -> attn.proj
        for s in ['weight', 'bias']:
            k = f'{p_in}self_attn.out_proj.{s}'
            if k in state_in:
                state_out[f'{p_out}attn.proj.{s}'] = state_in[k]

        # norm1 -> norm1, norm2 -> norm2 (PyTorch TransformerEncoderLayer)
        for norm_name in ['norm1', 'norm2']:
            for s in ['weight', 'bias']:
                k = f'{p_in}{norm_name}.{s}'
                if k in state_in:
                    state_out[f'{p_out}{norm_name}.{s}'] = state_in[k]

        # linear1 -> mlp.fc1, linear2 -> mlp.fc2
        for src, dst in [('linear1', 'mlp.fc1'), ('linear2', 'mlp.fc2')]:
            for s in ['weight', 'bias']:
                k = f'{p_in}{src}.{s}'
                if k in state_in:
                    state_out[f'{p_out}{dst}.{s}'] = state_in[k]

    # encoder_norm -> norm
    for s in ['weight', 'bias']:
        k = f'encoder_norm.{s}'
        if k in state_in:
            state_out[f'norm.{s}'] = state_in[k]

    torch.save({'model': state_out}, output_path)
    print(f'  Convertito: {os.path.basename(input_path)} -> {os.path.basename(output_path)}')
    print(f'  Chiavi output: {len(state_out)}')
    return output_path


# Converti tutti i checkpoint
for pct in MASK_RATIOS:
    if pct not in checkpoints_pretrained:
        continue
    out_path = os.path.join(RESULTS_DIR, f'mae_ast_{pct}pct_converted.pth')
    if os.path.exists(out_path):
        print(f'  {pct}% gia convertito: {out_path}')
    else:
        convert_simple_to_timm(checkpoints_pretrained[pct], out_path)
    checkpoints_converted[pct] = out_path

print(f'\nCheckpoint convertiti: {len(checkpoints_converted)}')

In [ ]:
# CELLA 3 — Scarica ESC-50
import glob

ESC50_DIR = '/content/ESC-50'
CSV_PATH  = f'{ESC50_DIR}/meta/esc50.csv'
AUDIO_DIR = f'{ESC50_DIR}/audio'

if os.path.exists(CSV_PATH):
    print('ESC-50 gia presente')
else:
    print('Scarico ESC-50 (~700 MB)...')
    os.system(f'git clone --depth 1 https://github.com/karolpiczak/ESC-50.git {ESC50_DIR} --quiet')

n_file = len(glob.glob(f'{AUDIO_DIR}/*.wav'))
print(f'File audio: {n_file}/2000')

import pandas as pd
df = pd.read_csv(CSV_PATH)
print(f'Classi: {df["category"].nunique()}')

In [ ]:
# CELLA 4 — Configurazione fine-tuning
import yaml

with open(f'{REPO_DIR}/config/finetune.yaml') as f:
    yaml_config = yaml.safe_load(f)

CONFIG = {
    'num_classes':       yaml_config['model']['num_classes'],
    'drop_rate':         yaml_config['model']['drop_rate'],
    'n_epochs':          yaml_config['training']['n_epochs'],
    'batch_size':        yaml_config['training']['batch_size'],
    'num_workers':       yaml_config['training']['num_workers'],
    'val_ogni':          yaml_config['training']['val_ogni'],
    'lr_encoder':        yaml_config['optimizer']['lr_encoder'],
    'lr_classificatore': yaml_config['optimizer']['lr_classificatore'],
    'lr_min':            yaml_config['optimizer']['lr_min'],
    'weight_decay':      yaml_config['optimizer']['weight_decay'],
    'label_smoothing':   yaml_config['optimizer']['label_smoothing'],
}
print('Configurazione fine-tuning:')
for k, v in CONFIG.items():
    print(f'  {k:<25} = {v}')

In [ ]:
# CELLA 5 — Fine-tuning 5-fold per ogni masking ratio
from src.train import cross_validation_5fold

risultati_partA = {}

for pct in MASK_RATIOS:
    if pct not in checkpoints_converted:
        print(f'\nSalto {pct}%: checkpoint non disponibile')
        continue

    ck_path = checkpoints_converted[pct]
    print(f'\n{"="*60}')
    print(f'  FINE-TUNING — Masking ratio {pct}%')
    print(f'  Checkpoint: {os.path.basename(ck_path)}')
    print(f'{"="*60}')

    risultati = cross_validation_5fold(
        checkpoint_path = ck_path,
        csv_path        = CSV_PATH,
        audio_dir       = AUDIO_DIR,
        config          = CONFIG,
        device          = DEVICE,
    )

    risultati_partA[pct] = risultati

    # Salva risultati per questo masking ratio
    out_path = os.path.join(RESULTS_DIR, f'partA_finetune_{pct}pct.json')
    with open(out_path, 'w') as f:
        json.dump({
            'masking_ratio': pct / 100,
            'accuracies': risultati['accuracies'],
            'media': risultati['media'],
            'std': risultati['std'],
            'config': CONFIG,
        }, f, indent=2)
    print(f'Risultati salvati: {out_path}')

In [ ]:
# CELLA 6 — Confronto accuracy 25% vs 50% vs 75%
import matplotlib.pyplot as plt
import numpy as np

print(f'\n{"="*60}')
print(f'  RISULTATI PARTE A — Confronto masking ratio')
print(f'{"="*60}')
print(f'  {"Masking %":<15} {"Accuracy":>12} {"Std":>8}')
print(f'  {"-"*40}')

medie = []
stds = []
labels = []

for pct in MASK_RATIOS:
    if pct not in risultati_partA:
        continue
    r = risultati_partA[pct]
    print(f'  {pct}%{" ":>13} {r["media"]*100:.2f}%     {r["std"]*100:.2f}%')
    medie.append(r['media'] * 100)
    stds.append(r['std'] * 100)
    labels.append(f'{pct}%')

# Verifica trend
if len(medie) == 3:
    trend_ok = medie[2] > medie[1] > medie[0]
    print(f'\n  Trend acc_75 > acc_50 > acc_25: {"CONFERMATO" if trend_ok else "NON CONFERMATO"}')

print(f'{"="*60}')

# Grafico
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy media per masking ratio
ax = axes[0]
colori = ['#E74C3C', '#F39C12', '#2E86C1']
bars = ax.bar(labels, medie, yerr=stds, color=colori, alpha=0.85, capsize=5)
for bar, m in zip(bars, medie):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{m:.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('Masking Ratio')
ax.set_title('Accuracy ESC-50 per masking ratio')
ax.grid(axis='y', alpha=0.3)

# Accuracy per fold
ax2 = axes[1]
x = np.arange(5)
w = 0.25
for i, pct in enumerate(MASK_RATIOS):
    if pct not in risultati_partA:
        continue
    accs = [a * 100 for a in risultati_partA[pct]['accuracies']]
    ax2.bar(x + i*w - w, accs, w, label=f'{pct}%', color=colori[i], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy per fold e masking ratio')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
graph_path = os.path.join(RESULTS_DIR, 'partA_finetune_comparison.png')
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico salvato: {graph_path}')